# Week 2 · Day 1 — Pandas fundamentals (DataFrames)

*A spreadsheet you can automate.*

**By the end you'll have shipped:** load a **coffee-orders table**, filter it, and produce a **sales summary by category** — in about five lines.

> Core Path = everything unmarked. `Go Deeper 🔧` = optional.
> Builds directly on **Week 1 Day 4 (NumPy)** — a DataFrame column *is* a NumPy array with a name.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1 · Python foundations (Week 2) |
| **Prerequisites** | Week 1 (esp. Day 4 — NumPy arrays) |
| **Est. time** | ~30 min |
| **Capstone tie-in** | *Matter Intelligence* — the table we filter/aggregate is what later lands in Snowflake |
| **Difficulty** | Core (+ optional Go Deeper) |

### 🎯 Learning objectives

By the end you'll be able to:
- Explain what a **DataFrame** is (a smart, programmable spreadsheet).
- **Create** one from a list of dicts and **read** one from a CSV file.
- **Select** columns and **filter** rows with a condition.
- **Sort** and **aggregate** with `groupby` — and see how these map to **SQL** you'll teach next.

### ⚖️ Why it matters

Last week you totalled a column of coffee **prices** with NumPy. But real data isn't one bare column — it's a **table**: each order has an item, a size, a store, a payment type. **pandas** gives you that whole table *in code*, with **named** columns you can `select`, `filter`, `sort`, and `group`.

Here's the key link: **a DataFrame is just a bundle of NumPy arrays with row and column labels.** `df["price"].values` hands you back the exact array from Day 4. So everything you learned — vectorized math, boolean masks — still works; pandas just makes it readable.

And the four moves today — **select, filter, sort, group** — are the exact moves of **SQL** (`SELECT`, `WHERE`, `ORDER BY`, `GROUP BY`). Learn them here and SQL will feel familiar when we reach Snowflake.

### ⚙️ Setup

Imports pandas and makes sure a sample `coffee_orders.csv` is reachable — it uses the shared `Training/data/coffee_orders.csv` if it can find it, otherwise it writes a local copy from a small built-in sample. So the lesson runs offline no matter where you launched Jupyter.

In [ ]:
import os
import pandas as pd

# A small built-in sample (same shape as the shared CSV) — used only as a fallback.
SAMPLE = [
    {"order_id":"O-5001","date":"2026-03-02","item":"Latte","size":"M","category":"Espresso Drink","price":4.75,"payment":"Card","store":"Downtown"},
    {"order_id":"O-5002","date":"2026-03-02","item":"Cold Brew","size":"L","category":"Cold","price":5.35,"payment":"App","store":"Airport"},
    {"order_id":"O-5003","date":"2026-03-03","item":"Drip","size":"S","category":"Brewed","price":2.50,"payment":"Cash","store":"Uptown"},
    {"order_id":"O-5004","date":"2026-03-03","item":"Mocha","size":"L","category":"Espresso Drink","price":5.95,"payment":"Card","store":"Downtown"},
    {"order_id":"O-5005","date":"2026-03-04","item":"Croissant","size":"","category":"Food","price":3.25,"payment":"App","store":"Airport"},
    {"order_id":"O-5006","date":"2026-03-04","item":"Cappuccino","size":"M","category":"Espresso Drink","price":4.50,"payment":"Card","store":"Uptown"},
    {"order_id":"O-5007","date":"2026-03-05","item":"Latte","size":"L","category":"Espresso Drink","price":5.50,"payment":"Cash","store":"Downtown"},
    {"order_id":"O-5008","date":"2026-03-05","item":"Muffin","size":"","category":"Food","price":2.95,"payment":"Card","store":"Airport"},
    {"order_id":"O-5009","date":"2026-03-06","item":"Cold Brew","size":"M","category":"Cold","price":4.65,"payment":"App","store":"Uptown"},
    {"order_id":"O-5010","date":"2026-03-06","item":"Espresso","size":"S","category":"Espresso Drink","price":2.75,"payment":"Cash","store":"Downtown"},
]

CSV_PATH = "coffee_orders.csv"                                   # local to this notebook
SHARED = os.path.join("..", "..", "data", "coffee_orders.csv")   # Training/data/coffee_orders.csv
if os.path.exists(SHARED):
    CSV_PATH = SHARED
elif not os.path.exists(CSV_PATH):
    pd.DataFrame(SAMPLE).to_csv(CSV_PATH, index=False)

print(f"pandas {pd.__version__} — using CSV: {CSV_PATH}")

### 1 · A DataFrame is a smart spreadsheet

A **DataFrame** is a table: rows (orders) and named columns (fields). We can build one straight from a list-of-dicts, just like the shape you met in Week 1 Day 2.

In [ ]:
df = pd.DataFrame(SAMPLE)

df.head()          # first 5 rows, nicely rendered (run this cell to see the table)

In [ ]:
print(f"shape (rows, cols): {df.shape}")
print(f"columns: {list(df.columns)}")

# a single column is a Series — and it's backed by a NumPy array (hello, Day 4!)
print(f'price column is a {type(df["price"]).__name__} -> .values is a {type(df["price"].values).__name__}')
df.info()          # dtypes + non-null counts — your first data-quality glance

**What just happened:** `df.head()` shows the top rows, `df.shape` gives (rows, columns), and `df.info()` reports each column's type. Notice `df["price"].values` is a NumPy array — the DataFrame is arrays + labels.

### 2 · Read a table from a CSV

In real life the data lives in a file. `pd.read_csv` loads it in one line — this is how most analyses start.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} orders from {CSV_PATH}")
df.head()

### 3 · Select columns  →  like SQL `SELECT`

Grab one column (a **Series**) with `df["col"]`, or several with a list of names. Picking columns is exactly what SQL's `SELECT` does.

In [ ]:
# one column
print(df["item"].head(3))
print("-" * 30)

# several columns  (SQL: SELECT item, category, price)
df[["item", "category", "price"]].head()

### 4 · Filter rows  →  like SQL `WHERE`

Write a **condition** to keep only the rows you want. `df["price"] > 5` produces True/False for every row (a NumPy mask, from Day 4!); putting it inside `df[ ... ]` keeps the True rows.

In [ ]:
# premium orders  (SQL: WHERE price > 5)
premium = df[df["price"] > 5]
print(f"{len(premium)} orders over $5")
premium[["order_id", "item", "price"]]

In [ ]:
# combine conditions with & (and) / | (or) — each condition needs ( )
# Downtown AND premium  (SQL: WHERE store='Downtown' AND price > 5)
downtown_premium = df[(df["store"] == "Downtown") & (df["price"] > 5)]
downtown_premium[["order_id", "item", "store", "price"]]

### 5 · Sort  →  like SQL `ORDER BY`

`sort_values` orders the table by a column.

In [ ]:
# priciest orders first  (SQL: ORDER BY price DESC)
df.sort_values("price", ascending=False)[["order_id", "item", "price"]].head()

### 6 · Aggregate with groupby  →  like SQL `GROUP BY`

The big one. **Group** rows by a category, then compute a number per group — total revenue by drink category, count of orders per store, and so on. This is `GROUP BY` in SQL, and it's where data work gets powerful.

In [ ]:
# total revenue per category  (SQL: SELECT category, SUM(price) ... GROUP BY category)
by_cat = df.groupby("category")["price"].sum().sort_values(ascending=False)
print(by_cat)

In [ ]:
# count of orders per store  (SQL: SELECT store, COUNT(*) ... GROUP BY store)
df.groupby("store").size().sort_values(ascending=False)

> **🔗 Bridge to SQL (what you'll teach next).** Every pandas move you just made has a one-to-one SQL twin:
>
> | Goal | pandas | SQL |
> |---|---|---|
> | pick columns | `df[["a","b"]]` | `SELECT a, b` |
> | keep some rows | `df[df.x > 5]` | `WHERE x > 5` |
> | order | `df.sort_values("x")` | `ORDER BY x` |
> | subtotal by group | `df.groupby("g")["x"].sum()` | `GROUP BY g` |
>
> Same thinking, two dialects. pandas is the training wheels; Snowflake SQL is the same ride at scale.

> **`Go Deeper 🔧` — `.loc`, computed columns, `value_counts()`.** `.loc[rows, cols]` selects by label; you can add a computed column in one line; and `value_counts()` is a fast category tally.

In [ ]:
# 1) add a computed column (vectorized — it's a NumPy op under the hood)
df["size_tier"] = df["price"].apply(lambda x: "premium" if x >= 5 else "standard")

# 2) .loc selects rows (by condition) and columns (by name) together
print(df.loc[df["size_tier"] == "premium", ["order_id", "item", "price"]])

# 3) quick category tally
print("\nOrders by payment type:")
print(df["payment"].value_counts())

> **`Common pitfalls ⚠️`**
>
> - Combine filters with `&` / `|` (not `and`/`or`), and wrap **each** condition in parentheses.
> - `=` assigns; `==` compares — filters use `==`.
> - Column names are case-sensitive and must match exactly (`"price"`, not `"Price"`).
> - A `SettingWithCopyWarning` usually means you filtered then assigned — create with `.copy()` if you plan to edit a subset.

### ✍️ Your turn

In [ ]:
# Using df:
# TODO 1: show only orders paid with the 'App'
# TODO 2: from those, select just order_id, item, price
# TODO 3: compute the AVERAGE price per category (hint: .mean())
# TODO 4 (stretch): how many orders did each store take? (hint: groupby + .size())

# your code here


<details><summary>✅ Show solution</summary>

```python
# 1 & 2
app_orders = df[df["payment"] == "App"]
print(app_orders[["order_id", "item", "price"]])

# 3
print(df.groupby("category")["price"].mean().round(2))

# 4
print(df.groupby("store").size())
```
</details>

### 🚀 Build the artifact — a sales summary by category

The full pipeline in a handful of lines: **read → filter → group → save**. This is a real, repeatable report — the kind of thing a manager would want every morning.

In [ ]:
# 1. read
df = pd.read_csv(CSV_PATH)

# 2. filter to drinks only (exclude the Food category)
drinks = df[df["category"] != "Food"]

# 3. group: total & average price by category, plus an order count
summary = (
    drinks.groupby("category")
          .agg(orders=("order_id", "count"),
               total_revenue=("price", "sum"),
               avg_price=("price", "mean"))
          .sort_values("total_revenue", ascending=False)
          .round(2)
)
print(summary)

# 4. save the report for sharing
summary.to_csv("sales_summary_by_category.csv")
print("\n✅ Shipped: sales_summary_by_category.csv")

> **🔗 Your world — from coffee to matters.** This pipeline *is* your billing report. Swap the table: `coffee_orders.csv` → `matters.csv`, `price` → **`amount_billed`**, `category` → **`practice_area`**. Then `df.groupby("practice_area")["amount_billed"].sum()` is your **billing summary by practice area** — the same five lines, the same `GROUP BY`. The coffee shop is just a friendlier place to learn the move.

### 📝 Recap — what you shipped

- A **DataFrame** is a programmable spreadsheet — a bundle of NumPy arrays with labels.
- **Select** columns, **filter** rows (with `&`/`|`), **sort**, and **group** to aggregate.
- Those four moves map directly to SQL `SELECT` / `WHERE` / `ORDER BY` / `GROUP BY`.
- **Artifact:** a saved sales-summary-by-category report.

### 🧠 Check your understanding

1. Which pandas operation is the twin of SQL's `WHERE`?
2. Why do combined filters need `&`/`|` and parentheses instead of `and`/`or`?
3. What does `df.groupby("category")["price"].sum()` give you?

<details><summary>Answers</summary>

1. **Filtering** rows with a condition: `df[df["x"] > 5]`.
2. pandas evaluates the condition across the whole column at once (element-wise, like a NumPy mask); `&`/`|` do that, and parentheses fix the order of operations.
3. The **total revenue for each category** — one subtotal per group.
</details>

### ➡️ Next up — Week 2, Day 2: cleaning & transforming

Real data is rarely this tidy. Next lesson we take a **messy** export — missing prices, `"$4.50"` text, `latte` vs `LATTE` — and clean it into something you can actually group and sum. Then Day 3 covers **grouping & joins**, and Day 4 rebuilds today's report in the faster **Polars** engine.

### 📖 Reference & glossary

| Term | Plain meaning | SQL twin |
|---|---|---|
| DataFrame | a programmable table | a table |
| Series | one column (a NumPy array + label) | one column |
| `df[[...]]` | pick columns | `SELECT` |
| `df[df.x > n]` | keep matching rows | `WHERE` |
| `sort_values` | order rows | `ORDER BY` |
| `groupby(...).agg(...)` | subtotal by category | `GROUP BY` |

**Official docs:** [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) · [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [`groupby`](https://pandas.pydata.org/docs/user_guide/groupby.html)